# Load dim_region

dim_region is derived from bronze.products data.

Starts with `%run "../../libs/notebook_init"` — see
`.claude/project/helpers.md` for what `notebook_init` injects (CATALOG,
BRONZE, SILVER, GOLD, AUDIT, RAW_FILES, STATUS_*, PIPELINE_RUN_ID, Utils, F,
datetime, etc.).


In [0]:
%run "../../libs/notebook_init"

In [ ]:
# Imports and constants specific to the dim_region Silver load.
# Source is bronze.products (distinct province/region triples); target is
# the SILVER.dim_region Type-1 dimension.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

# transform_detail_log_insert isn't in notebook_init's central import yet —
# pull it in here so this notebook can log per-transform audit rows.
from pipeline_logging import transform_detail_log_insert

SOURCE_SUBPATH = "masterdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{SILVER}.dim_region"

# print(f"Products Source Path:  {SOURCE_PATH}")


In [0]:
# ----------------------------------------------------------------------
# Setup the variables need for initial call to pipeline_step_log_upsert
# ----------------------------------------------------------------------

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

#logger.info(f"Inserting pipeline_step_log record for notebook {notebook_name}")

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = TARGET_TABLE
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

# All Parameters
# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)

In [ ]:
# ============================================================
#  Pipeline: {BRONZE}.products → {SILVER}.dim_region
#  Pattern: idempotent MERGE on natural key (Province, RegionName,
#           SubRegionName) — Type 1, no SCD2 history.
#  Audit  : Pattern B — transform_detail_log on success and failure.
# ============================================================

# Per-transform variables for transform_detail_log. Defined OUTSIDE the try
# block so the except handler can log a failed transform row even if the
# pre-count SQL never started. See .claude/project/gotchas.md
# "Audit-logging variables must be declared OUTSIDE the `try` block".
transform_source_table = f"{BRONZE}.products"
transform_target_table = f"{SILVER}.dim_region"
transform_started      = datetime.now(timezone.utc)
rows_inserted          = 0

try:
    # Pre-counts so we can derive rows_inserted via pre/post differential.
    # MERGE returns metrics via DESCRIBE HISTORY for non-ATOMIC blocks, but
    # we use the differential pattern here to match slvr_02 cell 5 and
    # remain robust to future refactors.
    pre_total_count = spark.table(transform_target_table).count()
    rows_read = spark.sql(f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT
                p.province,
                p.region_1,
                p.region_2
            FROM {BRONZE}.products p
            WHERE p.province IS NOT NULL
        )
    """).collect()[0][0]

    sql_query = f"""
         MERGE INTO {SILVER}.dim_region a
        USING (
            SELECT DISTINCT
                p.province       AS Province,
                p.region_1       AS RegionName,
                p.region_2       AS SubRegionName,
                current_timestamp() AS InsertedDate,
                current_timestamp() AS UpdatedDate
            FROM {BRONZE}.products p
            WHERE p.province IS NOT NULL
        ) t
        ON  a.Province = t.Province
        AND a.RegionName <=> t.RegionName
        AND a.SubRegionName <=> t.SubRegionName

        WHEN NOT MATCHED THEN
            INSERT (
                Province,
                RegionName,
                SubRegionName,
                InsertedDate,
                UpdatedDate
            )
            VALUES (
                t.Province,
                t.RegionName,
                t.SubRegionName,
                t.InsertedDate,
                t.UpdatedDate
            )
    """

    spark.sql(sql_query)

    # Post-count → rows_inserted via differential.
    post_total_count = spark.table(transform_target_table).count()
    rows_inserted    = post_total_count - pre_total_count
    rows_written     = rows_inserted

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    transform_detail_log_insert(
        spark,
        pipeline_run_id   = PIPELINE_RUN_ID,
        step_log_id       = step_log_id,
        source_table      = transform_source_table,
        target_table      = transform_target_table,
        status            = status,
        started_timestamp = transform_started,
        ended_timestamp   = ended_timestamp,
        rows_read         = rows_read,
        rows_written      = rows_written,
        rows_inserted     = rows_inserted,
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

    print(f"Success: dim_region was loaded. rows_inserted={rows_inserted}, rows_read={rows_read}")

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED

    transform_detail_log_insert(
        spark,
        pipeline_run_id   = PIPELINE_RUN_ID,
        step_log_id       = step_log_id,
        source_table      = transform_source_table,
        target_table      = transform_target_table,
        status            = status,
        started_timestamp = transform_started,
        ended_timestamp   = ended_timestamp,
        rows_inserted     = rows_inserted,
        error_message     = error_message,
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise


In [ ]:
%skip

metrics = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.dim_region LIMIT 1") \
               .select("operationMetrics") \
               .collect()[0][0]

rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
rows_deleted  = int(metrics.get("numTargetRowsDeleted",  0))

print(f" Rows  Inserted: {rows_inserted:,}")
print(f" Rows     Updated: {rows_updated:,}"   )
